# 07 — Carga Financeira e Consumo de Leitos

Insuficiência respiratória é uma condição cara: alta intensidade de UTI, estadias prolongadas, procedimentos complexos. Este notebook quantifica o **impacto financeiro e operacional** da J96 no SUS-SP.

**Pergunta de Pesquisa:** RQ6 — Qual é a carga financeira e de leitos?

1. Custo total, por internação e por leito-dia
2. Evolução temporal dos custos (acima da inflação?)
3. Custo × mortalidade (gastar mais = salvar mais?)
4. Consumo de leitos-dia (total e UTI)
5. Custo da mortalidade excedente: quanto custa o excesso de mortes?

**Depende de:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `hospital_icu_beds.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_icu_beds,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
icu_beds = load_icu_beds()

resp = resp.merge(tags[["CNES", "icu_capacity_level", "nat_jur_category"]], on="CNES", how="left")
resp = resp.merge(icu_beds[["CNES", "icu_beds_sus"]], on="CNES", how="left")
resp["icu_beds_sus"] = resp["icu_beds_sus"].fillna(0)
resp["has_icu"] = (resp["icu_beds_sus"] > 0).astype(int)
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])
resp["cost_per_day"] = resp["VAL_TOT"] / resp["DIAS_PERM"].clip(lower=1)

n_years = resp["year"].nunique()
print(f"Admissions: {len(resp):,}")
print(f"Total cost: R$ {resp['VAL_TOT'].sum():,.0f}")
print(f"Total bed-days: {resp['DIAS_PERM'].sum():,.0f}")
print(f"Total ICU-days: {resp['icu_days'].sum():,.0f}")

Admissions: 116,374
Total cost: R$ 383,471,537
Total bed-days: 1,125,335
Total ICU-days: 407,892


## 1. Visão Geral: Custos e Leitos

In [2]:
total_cost = resp["VAL_TOT"].sum()
total_bed_days = resp["DIAS_PERM"].sum()
total_icu_days = resp["icu_days"].sum()

# By outcome
survived = resp[resp["MORTE"] == 0]
died = resp[resp["MORTE"] == 1]

summary = pd.DataFrame({
    "Métrica": ["Internações", "Óbitos", "Custo Total (R$)", "Custo/internação (R$)",
                "Custo/leito-dia (R$)", "Leitos-dia totais", "UTI-dias totais",
                "Permanência média (dias)", "UTI média (dias)", "% UTI usada"],
    "Total": [f"{len(resp):,}", f"{resp['MORTE'].sum():,}",
              f"R$ {total_cost:,.0f}", f"R$ {resp['VAL_TOT'].mean():,.0f}",
              f"R$ {total_cost / total_bed_days:,.0f}", f"{total_bed_days:,.0f}",
              f"{total_icu_days:,.0f}", f"{resp['DIAS_PERM'].mean():.1f}",
              f"{resp['icu_days'].mean():.1f}", f"{(resp['icu_used'].mean()*100):.1f}%"],
    "Sobreviventes": [f"{len(survived):,}", "—",
                       f"R$ {survived['VAL_TOT'].sum():,.0f}", f"R$ {survived['VAL_TOT'].mean():,.0f}",
                       f"R$ {survived['VAL_TOT'].sum() / survived['DIAS_PERM'].clip(lower=1).sum():,.0f}",
                       f"{survived['DIAS_PERM'].sum():,.0f}",
                       f"{survived['icu_days'].sum():,.0f}",
                       f"{survived['DIAS_PERM'].mean():.1f}",
                       f"{survived['icu_days'].mean():.1f}",
                       f"{(survived['icu_used'].mean()*100):.1f}%"],
    "Óbitos": [f"{len(died):,}", f"{len(died):,}",
               f"R$ {died['VAL_TOT'].sum():,.0f}", f"R$ {died['VAL_TOT'].mean():,.0f}",
               f"R$ {died['VAL_TOT'].sum() / died['DIAS_PERM'].clip(lower=1).sum():,.0f}",
               f"{died['DIAS_PERM'].sum():,.0f}",
               f"{died['icu_days'].sum():,.0f}",
               f"{died['DIAS_PERM'].mean():.1f}",
               f"{died['icu_days'].mean():.1f}",
               f"{(died['icu_used'].mean()*100):.1f}%"],
})
print(summary.to_string(index=False))

                 Métrica          Total  Sobreviventes         Óbitos
             Internações        116,374         77,990         38,384
                  Óbitos         38,384              —         38,384
        Custo Total (R$) R$ 383,471,537 R$ 278,391,169 R$ 105,080,368
   Custo/internação (R$)       R$ 3,295       R$ 3,570       R$ 2,738
    Custo/leito-dia (R$)         R$ 341         R$ 334         R$ 352
       Leitos-dia totais      1,125,335        832,828        292,507
         UTI-dias totais        407,892        299,917        107,975
Permanência média (dias)            9.7           10.7            7.6
        UTI média (dias)            3.5            3.8            2.8
             % UTI usada         100.0%         100.0%         100.0%


## 2. Evolução Temporal dos Custos

In [3]:
annual = resp.groupby(resp["year"].astype(int)).agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    total_cost=("VAL_TOT", "sum"),
    mean_cost=("VAL_TOT", "mean"),
    median_cost=("VAL_TOT", "median"),
    total_bed_days=("DIAS_PERM", "sum"),
    mean_los=("DIAS_PERM", "mean"),
    total_icu_days=("icu_days", "sum"),
    mean_icu_days=("icu_days", "mean"),
).reset_index()
annual["cost_per_day"] = annual["total_cost"] / annual["total_bed_days"]

# IPCA approximate deflator (base 2016=100)
ipca = {2016: 100, 2017: 102.9, 2018: 106.7, 2019: 111.1, 2020: 115.7,
        2021: 127.4, 2022: 135.1, 2023: 141.3, 2024: 147.6, 2025: 153.5}
annual["ipca_idx"] = annual["year"].map(ipca)
annual["mean_cost_real"] = annual["mean_cost"] / annual["ipca_idx"] * 100
annual["cost_per_day_real"] = annual["cost_per_day"] / annual["ipca_idx"] * 100

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# A: Total cost by year
ax = axes[0, 0]
ax.bar(annual["year"], annual["total_cost"] / 1e6, color=COLORS["primary"], alpha=0.7)
ax.set_ylabel("Custo Total (R$ milhões)")
ax.set_title("A. Custo Total Anual (nominal)", fontweight="bold")
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")

# B: Mean cost per admission (nominal vs real)
ax = axes[0, 1]
ax.plot(annual["year"], annual["mean_cost"], "o-", color=COLORS["primary"], linewidth=2, label="Nominal")
ax.plot(annual["year"], annual["mean_cost_real"], "s--", color=COLORS["success"], linewidth=2, label="Real (IPCA 2016)")
ax.set_ylabel("Custo Médio por Internação (R$)")
ax.set_title("B. Custo Médio por Internação", fontweight="bold")
ax.legend()
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")

# C: Bed-days and ICU-days
ax = axes[1, 0]
ax.bar(annual["year"], annual["total_bed_days"] / 1000, color=COLORS["primary"], alpha=0.5, label="Leitos-dia")
ax.bar(annual["year"], annual["total_icu_days"] / 1000, color=COLORS["danger"], alpha=0.7, label="UTI-dias")
ax.set_ylabel("Dias (milhares)")
ax.set_title("C. Consumo de Leitos-Dia", fontweight="bold")
ax.legend()
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")

# D: Cost per bed-day (real)
ax = axes[1, 1]
ax.plot(annual["year"], annual["cost_per_day_real"], "D-", color=COLORS["warning"], linewidth=2.5, markersize=8)
ax.set_ylabel("R$ / leito-dia (real, IPCA 2016)")
ax.set_title("D. Custo por Leito-Dia (deflacionado)", fontweight="bold")
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")

fig.suptitle("Evolução Financeira: Insuficiência Respiratória (J96) no SUS-SP",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "cost_evolution", prefix="07")

# Growth rates
cost_2016 = annual[annual["year"] == 2016]["mean_cost_real"].values[0]
cost_last = annual[annual["year"] == annual["year"].max()]["mean_cost_real"].values[0]
print(f"Custo médio real (IPCA 2016):")
print(f"  2016: R$ {cost_2016:,.0f}")
print(f"  {int(annual['year'].max())}: R$ {cost_last:,.0f}")
print(f"  Variação real: {(cost_last/cost_2016 - 1)*100:+.1f}%")

  Saved: 07_cost_evolution.png
Custo médio real (IPCA 2016):
  2016: R$ 3,418
  2025: R$ 2,404
  Variação real: -29.7%


## 3. Custo por Faixa Etária e Desfecho

In [4]:
age_cost = resp.groupby(["age_group", "MORTE"], observed=True).agg(
    n=("VAL_TOT", "count"),
    mean_cost=("VAL_TOT", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    total_cost=("VAL_TOT", "sum"),
).reset_index()
age_cost["desfecho"] = age_cost["MORTE"].map({0: "Sobreviveu", 1: "Óbito"})

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# A: Mean cost by age and outcome
ax = axes[0]
pivot_cost = age_cost.pivot(index="age_group", columns="desfecho", values="mean_cost")
pivot_cost.plot(kind="bar", ax=ax, color=[COLORS["success"], COLORS["danger"]], alpha=0.7)
ax.set_ylabel("Custo Médio (R$)")
ax.set_title("A. Custo Médio por Idade × Desfecho", fontweight="bold")
ax.legend(title="")
ax.tick_params(axis="x", rotation=0)

# B: LOS by age and outcome
ax = axes[1]
pivot_los = age_cost.pivot(index="age_group", columns="desfecho", values="mean_los")
pivot_los.plot(kind="bar", ax=ax, color=[COLORS["success"], COLORS["danger"]], alpha=0.7)
ax.set_ylabel("Permanência Média (dias)")
ax.set_title("B. Permanência por Idade × Desfecho", fontweight="bold")
ax.legend(title="")
ax.tick_params(axis="x", rotation=0)

# C: Total cost share by age group
ax = axes[2]
age_total = resp.groupby("age_group", observed=True)["VAL_TOT"].sum()
age_total_pct = age_total / age_total.sum() * 100
colors_age = [COLORS["primary"], COLORS["primary"], COLORS["primary"],
              COLORS["warning"], COLORS["danger"], COLORS["danger"]]
ax.pie(age_total_pct, labels=[f"{g}\n({p:.0f}%)" for g, p in zip(age_total_pct.index, age_total_pct)],
       colors=colors_age, autopct="", startangle=90)
ax.set_title("C. Distribuição do Custo por Faixa Etária", fontweight="bold")

fig.suptitle("Custo por Faixa Etária e Desfecho", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "cost_by_age", prefix="07")

print("Custo médio por faixa e desfecho:")
for _, r in age_cost.iterrows():
    print(f"  {r['age_group']} {r['desfecho']}: R$ {r['mean_cost']:,.0f} | LOS: {r['mean_los']:.1f}d | N: {r['n']:,}")

  Saved: 07_cost_by_age.png
Custo médio por faixa e desfecho:
  <1 Sobreviveu: R$ 4,133 | LOS: 11.4d | N: 8,788
  <1 Óbito: R$ 5,197 | LOS: 9.9d | N: 262
  1-17 Sobreviveu: R$ 4,001 | LOS: 11.7d | N: 27,761
  1-17 Óbito: R$ 5,355 | LOS: 11.4d | N: 1,035
  18-39 Sobreviveu: R$ 3,810 | LOS: 10.2d | N: 7,361
  18-39 Óbito: R$ 3,385 | LOS: 7.0d | N: 2,094
  40-59 Sobreviveu: R$ 3,420 | LOS: 10.1d | N: 12,533
  40-59 Óbito: R$ 3,039 | LOS: 7.8d | N: 7,504
  60-74 Sobreviveu: R$ 3,039 | LOS: 10.0d | N: 13,445
  60-74 Óbito: R$ 2,968 | LOS: 8.0d | N: 13,362
  75+ Sobreviveu: R$ 2,280 | LOS: 8.7d | N: 7,802
  75+ Óbito: R$ 2,021 | LOS: 6.9d | N: 14,040


## 4. Custo × Mortalidade: Gastar Mais Salva Vidas?

In [5]:
# Hospital-level cost vs mortality
hosp_cost = resp.groupby("CNES").agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    mean_cost=("VAL_TOT", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mean_age=("age", "mean"),
    icu_beds=("icu_beds_sus", "first"),
    nat_jur=("nat_jur_category", "first"),
).reset_index()
hosp_cost = hosp_cost[hosp_cost["n"] >= 30].copy()

r_cost_mort, p_cost_mort = stats.spearmanr(hosp_cost["mean_cost"], hosp_cost["mortality"])

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# A: Cost vs mortality scatter
ax = axes[0]
ax.scatter(hosp_cost["mean_cost"], hosp_cost["mortality"] * 100,
           s=hosp_cost["n"] / 10, alpha=0.4, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.set_xlabel("Custo Médio por Internação (R$)")
ax.set_ylabel("Mortalidade (%)")
ax.set_title(f"A. Custo × Mortalidade (ρ = {r_cost_mort:.3f})", fontweight="bold")

# B: Cost by ICU level
ax = axes[1]
icu_cost = resp.groupby("icu_capacity_level", observed=True).agg(
    mean_cost=("VAL_TOT", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    n=("MORTE", "count"),
).reindex(["no_icu", "small_icu", "medium_icu", "large_icu"]).dropna()
icu_labels = {"no_icu": "Sem UTI", "small_icu": "Peq.", "medium_icu": "Méd.", "large_icu": "Grande"}
x_pos = range(len(icu_cost))
ax.bar(x_pos, icu_cost["mean_cost"], color=[COLORS["danger"], COLORS["warning"], COLORS["primary"], COLORS["success"]], alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{icu_labels.get(l, l)}\n(n={n:,})" for l, n in zip(icu_cost.index, icu_cost["n"])], fontsize=9)
ax.set_ylabel("Custo Médio (R$)")
ax.set_title("B. Custo por Nível de UTI", fontweight="bold")

# C: Cost by legal nature
ax = axes[2]
nat_cost = resp.groupby("nat_jur_category", observed=True).agg(
    mean_cost=("VAL_TOT", "mean"), n=("MORTE", "count"),
).sort_values("mean_cost", ascending=False)
nat_cost = nat_cost[nat_cost["n"] >= 100]
x_pos = range(len(nat_cost))
ax.barh(x_pos, nat_cost["mean_cost"], color=COLORS["secondary"], alpha=0.7)
ax.set_yticks(x_pos)
ax.set_yticklabels([f"{n} ({c:,})" for n, c in zip(nat_cost.index, nat_cost["n"])], fontsize=9)
ax.set_xlabel("Custo Médio (R$)")
ax.set_title("C. Custo por Natureza Jurídica", fontweight="bold")
ax.invert_yaxis()

fig.suptitle("Relação Custo × Mortalidade e Infraestrutura",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "cost_vs_mortality", prefix="07")

print(f"Correlação custo × mortalidade (hospital, n≥30): ρ = {r_cost_mort:.3f}, p = {p_cost_mort:.2e}")
print(f"\nCusto médio por nível UTI:")
for level, r in icu_cost.iterrows():
    print(f"  {icu_labels.get(level, level)}: R$ {r['mean_cost']:,.0f} | LOS: {r['mean_los']:.1f}d")

  Saved: 07_cost_vs_mortality.png
Correlação custo × mortalidade (hospital, n≥30): ρ = -0.112, p = 3.67e-02

Custo médio por nível UTI:
  Sem UTI: R$ 3,074 | LOS: 9.5d
  Peq.: R$ 3,445 | LOS: 9.4d
  Méd.: R$ 3,539 | LOS: 10.4d
  Grande: R$ 4,448 | LOS: 12.9d


## 5. Custo da Mortalidade Excedente Pós-COVID

In [6]:
# Pre-COVID mortality rate as baseline
pre_covid = resp[resp["year"].between(2016, 2019)]
post_covid = resp[resp["year"] >= 2022]

mort_pre = pre_covid["MORTE"].mean()
mort_post = post_covid["MORTE"].mean()
excess_mort_rate = mort_post - mort_pre
excess_deaths_post = excess_mort_rate * len(post_covid)

# Cost of excess deaths
mean_cost_death = died["VAL_TOT"].mean()
mean_cost_survivor = survived["VAL_TOT"].mean()
mean_los_death = died["DIAS_PERM"].mean()
mean_los_survivor = survived["DIAS_PERM"].mean()

# If excess deaths had survived instead:
cost_if_survived = excess_deaths_post * mean_cost_survivor
cost_actual = excess_deaths_post * mean_cost_death
cost_diff = cost_actual - cost_if_survived
bed_days_saved = excess_deaths_post * (mean_los_death - mean_los_survivor)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A: Pre vs post mortality/cost comparison
ax = axes[0]
categories = ["Mortalidade (%)", "Custo Médio (R$)", "Permanência (dias)"]
pre_vals = [mort_pre * 100, pre_covid["VAL_TOT"].mean(), pre_covid["DIAS_PERM"].mean()]
post_vals = [mort_post * 100, post_covid["VAL_TOT"].mean(), post_covid["DIAS_PERM"].mean()]

x = np.arange(3)
width = 0.35
# Normalize for comparison
pre_norm = [v / m for v, m in zip(pre_vals, pre_vals)]
post_norm = [v / m for v, m in zip(post_vals, pre_vals)]

ax.bar(x - width/2, pre_norm, width, label="Pré-COVID (2016-2019)", color=COLORS["primary"], alpha=0.7)
ax.bar(x + width/2, post_norm, width, label="Pós-COVID (2022-2025)", color=COLORS["danger"], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel("Índice (pré-COVID = 1,0)")
ax.set_title("A. Comparação Pré × Pós-COVID (normalizado)", fontweight="bold")
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.5)
ax.legend()

# B: Cost of excess deaths
ax = axes[1]
bars = ax.bar(["Excesso de\nóbitos", "Custo do\nexcesso (R$ mi)", "Leitos-dia\nexcedentes (mil)"],
              [excess_deaths_post, cost_diff / 1e6, bed_days_saved / 1000],
              color=[COLORS["danger"], COLORS["warning"], COLORS["primary"]], alpha=0.7)
ax.set_title("B. Impacto Financeiro da Mortalidade Excedente", fontweight="bold")
for bar, val in zip(bars, [f"{excess_deaths_post:,.0f}", f"R$ {cost_diff/1e6:,.1f}M", f"{bed_days_saved/1000:,.0f}k"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.02,
            val, ha="center", fontsize=10, fontweight="bold")

fig.suptitle("Custo do Excesso de Mortalidade Pós-COVID", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "excess_cost", prefix="07")

print(f"Mortalidade pré-COVID: {mort_pre*100:.1f}%")
print(f"Mortalidade pós-COVID: {mort_post*100:.1f}%")
print(f"Excesso pós-COVID: +{excess_mort_rate*100:.1f}pp")
print(f"Excesso de óbitos (2022-2025): {excess_deaths_post:,.0f}")
print(f"\nCusto médio: óbito R$ {mean_cost_death:,.0f} vs sobrevivente R$ {mean_cost_survivor:,.0f}")
print(f"Permanência: óbito {mean_los_death:.1f}d vs sobrevivente {mean_los_survivor:.1f}d")
print(f"\nSe mortalidade tivesse ficado no nível pré-COVID:")
print(f"  Leitos-dia economizados: {bed_days_saved:,.0f}")
n_post_years = post_covid["year"].nunique()
print(f"  Excesso anual pós-COVID: {excess_deaths_post / n_post_years:,.0f} óbitos/ano")

  Saved: 07_excess_cost.png
Mortalidade pré-COVID: 31.0%
Mortalidade pós-COVID: 35.1%
Excesso pós-COVID: +4.1pp
Excesso de óbitos (2022-2025): 1,881

Custo médio: óbito R$ 2,738 vs sobrevivente R$ 3,570
Permanência: óbito 7.6d vs sobrevivente 10.7d

Se mortalidade tivesse ficado no nível pré-COVID:
  Leitos-dia economizados: -5,751
  Excesso anual pós-COVID: 470 óbitos/ano


## 6. Métricas Consolidadas

In [7]:
metrics = {
    "total_cost_brl": round(total_cost),
    "annual_cost_brl": round(total_cost / n_years),
    "mean_cost_per_admission": round(resp["VAL_TOT"].mean()),
    "mean_cost_per_bed_day": round(total_cost / total_bed_days),
    "total_bed_days": int(total_bed_days),
    "total_icu_days": int(total_icu_days),
    "annual_bed_days": round(total_bed_days / n_years),
    "annual_icu_days": round(total_icu_days / n_years),
    "mean_los": round(resp["DIAS_PERM"].mean(), 1),
    "mean_los_survivor": round(mean_los_survivor, 1),
    "mean_los_death": round(mean_los_death, 1),
    "mean_cost_survivor": round(mean_cost_survivor),
    "mean_cost_death": round(mean_cost_death),
    "cost_mortality_rho": round(r_cost_mort, 3),
    "cost_real_change_pct": round((cost_last / cost_2016 - 1) * 100, 1),
    "excess_deaths_post_covid": round(excess_deaths_post),
    "excess_annual_post_covid": round(excess_deaths_post / n_post_years),
}
save_metrics(metrics, "07_financial_burden")
print("Metrics saved.")

  Saved metrics: 07_financial_burden.json
Metrics saved.
